In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

#Models from Scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import IsolationForest


#Model Evaluations 
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import confusion_matrix, classification_report 
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from sklearn.metrics import RocCurveDisplay

pd.set_option('display.max_columns', None)

## Load Dataset

In [2]:
def load_data():
    root_dir = Path.cwd().parent
    file_path = root_dir / "Dataset" / "diabetes_binary_health_indicators_BRFSS2015.csv"
    
    if not file_path.exists():
        raise FileNotFoundError(f"Dataset not found at: {file_path}")
    
    df = pd.read_csv(file_path)
    df.columns = df.columns.str.lower()
    return df

df = load_data()
df.head()

,diabetes_binary,highbp,highchol,cholcheck,bmi,smoker,stroke,heartdiseaseorattack,physactivity,fruits,veggies,hvyalcoholconsump,anyhealthcare,nodocbccost,genhlth,menthlth,physhlth,diffwalk,sex,age,education,income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


## Train / Validation / Test Split

In [3]:
def split_features_target(df):
    X = df.drop("diabetes_binary", axis=1)
    y = df["diabetes_binary"]
    return X, y

X, y = split_features_target(df)


In [4]:
def train_test_split_data(X, y, test_size=0.2, random_state=42):
    return train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

X_train, X_test, y_train, y_test = train_test_split_data(X, y)

# Scaling and Modelling

In [ ]:
def scale_data(X_train, X_test):
    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, scaler

X_train_scaled, X_test_scaled, scaler = scale_data(X_train, X_test)

#Put models into dictionary 
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN": KNeighborsClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "SVM": SVC(),

    # Anomaly Detection
    "Isolation Forest": IsolationForest(contamination=0.05, random_state=42)
}

# Create a function to fit and score models
def fit_and_score(models, X_train, X_test, y_train, y_test): 
    """
    Fits and evaluates given machine learning models.
    models: a dict of different Scikit-Learn machine learning models
    X_train: training data(with no labels)
    X_test: testing data(with no labels)
    y_train: training labels
    y_test: testing labels
    """
    
    #set random seed
    np.random.seed(42)

    # Making a dictionary to keep model scoresS
    model_scores = {}

    # Loop through models
    for name, model in models.items():

        # If anomaly detection model
        if name == "Isolation Forest":
            model.fit(X_train)

            # convert predictions (-1, 1) → (0, 1)
            y_pred = model.predict(X_test)
            y_pred = [1 if x == -1 else 0 for x in y_pred]

            # model_scores[name] = accuracy_score(y_test, y_pred)
            model_scores[name] = {
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred),
                "f1": f1_score(y_test, y_pred)
            }

        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            # model_scores[name] = model.score(X_test, y_test)
            model_scores[name] = {
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred),
                "f1": f1_score(y_test, y_pred)
            }

    return model_scores

model_scores = fit_and_score(models=models,
                             X_train=X_train_scaled,
                             X_test=X_test_scaled,
                             y_train=y_train,
                             y_test=y_test)

model_scores

In [ ]:
df_model_scores = pd.DataFrame(model_scores).T
print(df_model_scores.sort_values("f1", ascending=False))

Hyperparameter tuning with RandomizedSearchCV (CV= Cross Validated)
<br>
<br>#Put models into dictionary 
<br>models = {
    <br>"Logistic Regression": LogisticRegression(max_iter=1000),
    <br>"KNN": KNeighborsClassifier(),
    <br>"Random Forest": RandomForestClassifier(),
    <br>"Naive Bayes": GaussianNB(),
    <br>"Decision Tree": DecisionTreeClassifier(),
    <br>"Gradient Boosting": GradientBoostingClassifier(),
    <br>"SVM": SVC(),

    <br> # Anomaly Detection
    <br>"Isolation Forest": IsolationForest(contamination=0.05, random_state=42)
<br>}


In [ ]:
# hyperparamter grids
param_grids = {

    "Logistic Regression": {
        "C": np.logspace(-4, 4, 20),
        "solver": ['liblinear', 'lbfgs']
    },

    "KNN": {
        "n_neighbors": np.arange(3, 21, 2),
        "weights": ['uniform', 'distance'],
        "metric": ['euclidean', 'manhattan']
    },

    "Random Forest": {
        "n_estimators": np.arange(50, 200, 25),
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": np.arange(2, 10, 2),
        "min_samples_leaf": np.arange(1, 5, 1),
        "max_features": ['sqrt', 'log2']
    },

    "Naive Bayes": {
        "var_smoothing": np.logspace(0, -9, 100)
    },

    "Decision Tree": {
        "max_depth": [None, 5, 10, 20, 30],
        "min_samples_split": np.arange(2, 10, 2),
        "min_samples_leaf": np.arange(1, 5, 1),
        "criterion": ['gini', 'entropy']
    },

    "Gradient Boosting": {
        "n_estimators": np.arange(50, 200, 25),
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "max_depth": [3, 5, 10],
        "min_samples_split": np.arange(2, 10, 2),
        "min_samples_leaf": np.arange(1, 5, 1)
    },

    "SVM": {
        "C": np.logspace(-3, 3, 10),
        "gamma": ['scale', 'auto'],
        "kernel": ['linear', 'rbf', 'poly']
    },

    "Isolation Forest": {
        "n_estimators": np.arange(50, 200, 25),
        "max_samples": ['auto', 0.5, 0.75, 1.0],
        "contamination": [0.01, 0.05, 0.1],
        "max_features": np.arange(1, X_train.shape[1] + 1),
        "bootstrap": [True, False]
    }
}

## Tuning the models
def tune_all_models(models, param_grids, X_train, y_train, X_test, y_test):
    results = []

    for name, model in models.items():
        print(f"\n Tuning {name}")

        np.random.seed(42)

        # Handle Isolation Forest separately
        if name == "Isolation Forest":
            rs = RandomizedSearchCV(
                estimator=model,
                param_distributions=param_grids[name],
                n_iter=20,
                cv=3,
                verbose=1,
                n_jobs=-1
            )

            rs.fit(X_train)  # No y_train

            # Predict anomalies
            y_pred = best_model.predict(X_test)

            # Convert: -1 → 1 (anomaly), 1 → 0 (normal)
            y_pred = np.array([1 if x == -1 else 0 for x in y_pred])

            # Evaluate using labels
            test_score = f1_score(y_test, y_pred, zero_division=0)

            results.append({
                "Model": name,
                "Best Params": rs.best_params_,
                "Score": "N/A (unsupervised)"
            })

            print(f" Best Params: {rs.best_params_}")
            print(f"F1 Score: {test_score:.2f}")
            print("No test score (unsupervised model)")
            continue

        # Supervised models
        rs = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_grids[name],
            n_iter=20,
            cv=5,
            verbose=1,
            n_jobs=-1
        )

        rs.fit(X_train, y_train)

        best_params = rs.best_params_
        test_score = rs.score(X_test, y_test)

        results.append({
            "Model": name,
            "Best Params": best_params,
            "Score": test_score
        })

        print(f"Best Params: {best_params}")
        print(f"Test Score: {test_score:.2f}")

    return pd.DataFrame(results)

results_df = tune_all_models(
                             models, 
                             param_grids, 
                             X_train_scaled, 
                             y_train, 
                             X_test_scaled, 
                             y_test
                            )

# Sort by score (ignore Isolation Forest)
results_df_sorted = results_df.sort_values(by="Score", ascending=False, na_position='last')

print("\n Final Results:")
print(results_df_sorted)

Hyperparameter Tuning with GridSearchCV

--> check the result from above hyperparamter tuning, see which model has the best score 
<br>tesko matra hyperparamter tuning with gridsearchcv bata check garne..
<br> below randomforestclassifier is just an example

In [ ]:
#Different hyperparameter for our RandomForestClassifier model --> use same parameters from above
rf_grid = {"n_estimators":np.arange(10,80,8),
           "max_depth":[None,3,5,10],
           "min_samples_split":np.arange(2,10,2),
           "min_samples_leaf":np.arange(1,10,2)}

#Setup grid hyperparameter search for RandomForestClassifier

np.random.seed(42)

## change the model name accordingly
gs_rf = GridSearchCV(RandomForestClassifier(),
                           param_grid=rf_grid,
                           cv=10,
                           verbose=True,
                           n_jobs=4)
#Fit grid hyperparameter search model for RandomForestClassifer
gs_rf.fit(X_train, y_train)

In [ ]:
#Find the best hyperparameter for RandomForestClassifier
gs_rf.best_params_

In [ ]:
#Evaluate the Grid Search RandomForestClassifier
gs_rf.score(X_test_scaled,y_test)

In [ ]:
## ideal hyperparameter
ideal_model = RandomForestClassifier(n_estimators=18,
                                    min_samples_split=2,
                                    min_samples_leaf=7,
                                    max_depth=None,
                                    n_jobs=-1,
                                    random_state=42,
                                    max_samples=None)

In [ ]:
## Fit the model
ideal_model.fit(X_train_scaled, y_train)

In [ ]:
ideal_model.score(X_test_scaled, y_test)

Evaluating our tuned machine learning classifier, beyond accuracy
ROC curve and AUC curve

confusion matrix

classification report

Precision

Recall

F1-score

and it would be great if cross-validation is used where possible.

To make comparisons and evaluate our trained model, first we need to make predictions.

In [ ]:
# Make predictions with tuned models
y_predicts = gs_rf.predict(X_test_scaled)

In [ ]:
# Plot ROC curve and calculate the AUC(area under curve) metric
plot_roc_curve(gs_rf, X_test_scaled, y_test)

In [ ]:
#Confusion Matrix
print(confusion_matrix(y_test, y_predicts))

In [ ]:
sns.set(font_scale=1.5)

def plot_conf_mat(y_test, y_predicts):
  """
  Plot a confusion matrix using Seaborn's heatmap()
  """
  fig, ax = plt.subplots(figsize=(3,3))
  ax = sns.heatmap(confusion_matrix(y_test,y_predicts),
                  annot=True,
                   cbar=False)
  plt.xlabel("True label")
  plt.ylabel("Predicted label")

plot_conf_mat(y_test, y_predicts)

In [ ]:
print(classification_report(y_test, y_predicts))

Calculating the accuracy, precision, recall, and f1-score of our model using cross-validation[cross_val_score()]
Check best hyperparameters

In [ ]:
gs_rf.best_params_

In [ ]:
clf = RandomForestClassifier(max_depth=None,
                             min_samples_leaf=7,
                             min_samples_split=2,
                             n_estimators=18)

In [ ]:
## Cross-validated accuracy

cv_accuracy = cross_val_score(clf,
                         X,
                         y,
                         cv=5,
                         scoring="accuracy")
cv_accuracy 

In [ ]:
cv_accuracy = np.mean(cv_accuracy)
cv_accuracy

In [ ]:
## Cross-validated precision

cv_precision = cross_val_score(clf,
                               X,
                               y,
                               cv=5,
                               scoring="precision")
cv_precision = np.mean(cv_precision)
cv_precision

In [ ]:
## Cross-validated recall

cv_recall = cross_val_score(clf,
                               X,
                               y,
                               cv=5,
                               scoring="recall")
cv_recall = np.mean(cv_recall)
cv_recall

In [ ]:
## Cross-validated f1-score

cv_f1 = cross_val_score(clf,
                               X,
                               y,
                               cv=5,
                               scoring="f1")
cv_f1 = np.mean(cv_f1)
cv_f1

In [ ]:
# Visualizing the cross-validated metrics 


cv_metrics = pd.DataFrame({"Accuracy":cv_accuracy,
                           "Precision":cv_precision,
                           "Recall":cv_recall,
                           "F1":cv_f1},
                          index=[0])
cv_metrics.T.plot.bar(title="Cross-validated classification metrics",
                      legend=False)
plt.xticks(rotation=0)

In [ ]:
# Find the feature importance of our best model
ideal_model.feature_importances_

In [ ]:
## functions to plot the features importances
def plot_features(columns, importances, n=10):
    df = (pd.DataFrame({"features": columns,
                        "features_importance": importances})
         .sort_values("features_importance", ascending=False)
         .reset_index(drop=True))
    
    # Plot the dataframe
    fig, ax = plt.subplots()
    ax.barh(df["features"][:n], df["features_importance"][:10])
    ax.set_ylabel("features")
    ax.set_xlabel("feautures_importance")
    ax.invert_yaxis()

In [ ]:
plot_features(X_train.columns, ideal_model.feature_importances_)